### Basic Chat Bot with LangGraph (Graph API technology)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

#### add_messages are called reducers. Only used to append the vairables in the state

##### There are states in agentic AI. For saving the messages in the state. Appends the useful messages int the state.

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The 'add_messages' function
    # In the annotation defines how this state key should be updated
    # (in the case, it appends messages to the list, rather than overwriting them)

    messages: Annotated[list,add_messages]

graph_builder = StateGraph(State)

In [ ]:
graph_builder

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:qwen/qwen3.6-27b")

In [ ]:
def chatbot(state:State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("llm",chatbot)

# Adding Edges
graph_builder.add_edge(START,"llm")
graph_builder.add_edge("llm",END)

# Compile the graph
graph = graph_builder.compile()

In [ ]:
# Visualize the graph
from IPython.display import Image,display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages":"Hi"})

In [ ]:
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages":"Hi! How are you?"}):
    for value in event.values():
        print(value["messages"][-1].content)


### Chat Bot with Tool Calling [For better context purposes...]

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
tool = TavilySearch(max_results=2)

# Example usage
tool.invoke("What is langgraph?")

In [ ]:
# Custom Function
def multiply(a:int, b:int)->int:
    """
    Multiply a and b

    Args:
        a: first int
        b: second int

    Returns:
        int: output int
    """

    return a*b

In [ ]:
tools = [tool,multiply]

In [ ]:
llm_with_tool = llm.bind_tools(tools)

In [ ]:
# StateGraph: Already imported Start, End and State
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# Node Defination
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

In [ ]:
# Graph Builder
builder = StateGraph(State)

# Add Node
builder.add_node("tool_calling_llm",tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

# Add Edges
builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")
builder.add_edge("tools",END)

# Compile the graph
graph = builder.compile()


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
response = graph.invoke({"messages": "Give me the recent AI news and then multiply 5 by 10"})

for m in response['messages']:
    m.pretty_print()

### Adding Memory in Agentic Graph